# Commodity Payloads Deduplication & Consolidation

This notebook:
1. Loads all three commodity payloads (wheat, corn, soy)
2. Inspects their structure and identifies duplicates
3. Removes duplicates within and across payloads
4. Exports a consolidated, deduplicated dataset

## 1. Import Required Libraries

In [12]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Define data directory
DATA_DIR = Path('data')
print(f"Data directory: {DATA_DIR}")
print(f"Data directory exists: {DATA_DIR.exists()}")

Data directory: data
Data directory exists: True


## 2. Load All Three Payload Files

In [13]:
# Define payload file paths
payload_files = {
    'wheat': DATA_DIR / 'wheat' / 'wheat_payload.json',
    'corn': DATA_DIR / 'corn' / 'corn_payload.json',
    'soy': DATA_DIR / 'soy' / 'soy_payload.json',
}

# Check which files exist
print("Checking payload files:")
for commodity, path in payload_files.items():
    exists = path.exists()
    size = path.stat().st_size / 1024 if exists else 0
    print(f"  {commodity:8} -> {path.name:20} | Exists: {exists:5} | Size: {size:8.1f} KB")

Checking payload files:
  wheat    -> wheat_payload.json   | Exists:     1 | Size:     50.7 KB
  corn     -> corn_payload.json    | Exists:     1 | Size:   1608.9 KB
  soy      -> soy_payload.json     | Exists:     0 | Size:      0.0 KB


In [14]:
# Load all payloads
payloads = {}
original_counts = {}

for commodity, path in payload_files.items():
    if path.exists():
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            
            # Convert to DataFrame
            if isinstance(data, list):
                df = pd.DataFrame(data)
            elif isinstance(data, dict) and 'articles' in data:
                df = pd.DataFrame(data['articles'])
            else:
                df = pd.DataFrame([data])
            
            payloads[commodity] = df
            original_counts[commodity] = len(df)
            print(f"✓ Loaded {commodity:8} -> {len(df):6} records")
            
        except Exception as e:
            print(f"✗ Error loading {commodity}: {str(e)}")
    else:
        print(f"✗ File not found: {path}")

print(f"\nTotal payloads loaded: {len(payloads)}")

✓ Loaded wheat    ->     30 records
✓ Loaded corn     ->     30 records
✗ File not found: data/soy/soy_payload.json

Total payloads loaded: 2


## 3. Inspect Payload Structure

In [15]:
# Inspect each payload
for commodity, df in payloads.items():
    print(f"\n{'='*70}")
    print(f"  {commodity.upper()} Payload")
    print(f"{'='*70}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nMissing Values:\n{df.isnull().sum()}")
    print(f"\nFirst 2 rows:")
    print(df.head(2).to_string())


  WHEAT Payload
Shape: 30 rows × 14 columns

Columns: ['name', 'content', 'smlID', 'authorID', 'authorName', 'authorType', 'date', 'dateTimestamp', 'dataID', 'link', 'isEditorPick', 'image', 'searchable', 'prioirty']

Data Types:
name               str
content            str
smlID              str
authorID           str
authorName         str
authorType         str
date               str
dateTimestamp    int64
dataID             str
link               str
isEditorPick      bool
image              str
searchable         str
prioirty         int64
dtype: object

Missing Values:
name             0
content          0
smlID            0
authorID         0
authorName       0
authorType       0
date             0
dateTimestamp    0
dataID           0
link             0
isEditorPick     0
image            0
searchable       0
prioirty         0
dtype: int64

First 2 rows:
                            name                                                                                          

## 4. Identify and Remove Duplicates

In [16]:
# Identify common key columns for deduplication
all_columns = set()
for df in payloads.values():
    all_columns.update(df.columns)

print("All unique columns across payloads:")
print(sorted(all_columns))

# Define deduplication strategy
# Priority: URL > Title+Source > ID
dedup_keys = {
    'url': 'URL (most reliable)',
    'title_source': 'Title + Source',
    'id': 'ID',
}

print(f"\nDeduplication strategy: {dedup_keys}")

All unique columns across payloads:
['authorID', 'authorName', 'authorType', 'content', 'dataID', 'date', 'dateTimestamp', 'image', 'isEditorPick', 'link', 'name', 'prioirty', 'searchable', 'smlID']

Deduplication strategy: {'url': 'URL (most reliable)', 'title_source': 'Title + Source', 'id': 'ID'}


In [17]:
# Remove duplicates within each payload
deduplicated_payloads = {}
duplicate_counts = {}

for commodity, df in payloads.items():
    df_clean = df.copy()
    original_size = len(df_clean)
    
    # Strategy 1: Remove duplicates by URL
    if 'url' in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=['url'], keep='first')
    
    # Strategy 2: Remove duplicates by title + source (if URL not available)
    elif 'title' in df_clean.columns and 'source' in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=['title', 'source'], keep='first')
    
    # Remove rows with missing critical fields
    if 'url' in df_clean.columns:
        df_clean = df_clean.dropna(subset=['url'])
    
    deduplicated_payloads[commodity] = df_clean
    duplicate_counts[commodity] = original_size - len(df_clean)
    
    print(f"\n{commodity.upper()}:")
    print(f"  Original:      {original_size:6} records")
    print(f"  Deduplicated:  {len(df_clean):6} records")
    print(f"  Removed:       {duplicate_counts[commodity]:6} duplicates ({duplicate_counts[commodity]/original_size*100:.1f}%)")


WHEAT:
  Original:          30 records
  Deduplicated:      30 records
  Removed:            0 duplicates (0.0%)

CORN:
  Original:          30 records
  Deduplicated:      30 records
  Removed:            0 duplicates (0.0%)


In [18]:
# Check for cross-payload duplicates
print(f"\n{'='*70}")
print("  Cross-Payload Duplicate Analysis")
print(f"{'='*70}\n")

if len(deduplicated_payloads) > 1:
    # Collect all URLs
    url_counts = Counter()
    url_to_commodity = {}
    
    for commodity, df in deduplicated_payloads.items():
        if 'url' in df.columns:
            for url in df['url'].dropna():
                url_counts[url] += 1
                if url not in url_to_commodity:
                    url_to_commodity[url] = []
                url_to_commodity[url].append(commodity)
    
    # Find duplicates across payloads
    cross_dupes = {url: commodities for url, commodities in url_to_commodity.items() if len(set(commodities)) > 1}
    
    print(f"Total unique URLs: {len(url_counts)}")
    print(f"URLs appearing in multiple payloads: {len(cross_dupes)}")
    
    if cross_dupes:
        print(f"\nSample cross-payload duplicates (first 5):")
        for i, (url, commodities) in enumerate(list(cross_dupes.items())[:5]):
            print(f"  {i+1}. Appears in: {', '.join(commodities)}")
            print(f"     URL: {url[:70]}..." if len(url) > 70 else f"     URL: {url}")
else:
    print("Only one payload loaded, cannot check cross-payload duplicates.")


  Cross-Payload Duplicate Analysis

Total unique URLs: 0
URLs appearing in multiple payloads: 0


## 5. Consolidate and Export Clean Payload

In [19]:
# Combine all deduplicated payloads
if deduplicated_payloads:
    # Add commodity column to each dataframe
    dfs_with_commodity = []
    for commodity, df in deduplicated_payloads.items():
        df_copy = df.copy()
        df_copy['commodity'] = commodity
        dfs_with_commodity.append(df_copy)
    
    # Concatenate all dataframes
    df_consolidated = pd.concat(dfs_with_commodity, ignore_index=True)
    print(f"Consolidated dataset: {len(df_consolidated)} total records from {len(deduplicated_payloads)} commodities")
    
    # Remove duplicates across all payloads (by URL)
    if 'url' in df_consolidated.columns:
        df_consolidated = df_consolidated.drop_duplicates(subset=['url'], keep='first')
        print(f"After cross-payload dedup: {len(df_consolidated)} unique records")
    
    # Sort by commodity and date (if available)
    sort_cols = []
    if 'commodity' in df_consolidated.columns:
        sort_cols.append('commodity')
    if 'date' in df_consolidated.columns:
        df_consolidated['date'] = pd.to_datetime(df_consolidated['date'], errors='coerce')
        sort_cols.append('date')
    
    if sort_cols:
        df_consolidated = df_consolidated.sort_values(sort_cols, na_position='last').reset_index(drop=True)
        print(f"Sorted by: {', '.join(sort_cols)}")
    
    print(f"\nConsolidated DataFrame shape: {df_consolidated.shape}")
else:
    print("No payloads to consolidate.")

Consolidated dataset: 60 total records from 2 commodities
Sorted by: commodity, date

Consolidated DataFrame shape: (60, 15)


In [20]:
# Summary statistics
print(f"\n{'='*70}")
print("  DEDUPLICATION SUMMARY")
print(f"{'='*70}\n")

total_original = sum(original_counts.values())
total_deduplicated = sum(len(df) for df in deduplicated_payloads.values())
total_consolidated = len(df_consolidated)

print(f"Original records (sum of all payloads):  {total_original:8}")
print(f"After within-payload dedup:             {total_deduplicated:8} (-{total_original - total_deduplicated})")
print(f"After cross-payload dedup:              {total_consolidated:8} (-{total_deduplicated - total_consolidated})")
print(f"\nTotal removed: {total_original - total_consolidated} ({(total_original - total_consolidated) / total_original * 100:.1f}%)")

print(f"\nBreakdown by commodity:")
for commodity in sorted(original_counts.keys()):
    original = original_counts[commodity]
    removed = duplicate_counts.get(commodity, 0)
    pct = (removed / original * 100) if original > 0 else 0
    print(f"  {commodity:8} -> {original:6} → {original - removed:6} (removed {removed:4}, {pct:5.1f}%)")


  DEDUPLICATION SUMMARY

Original records (sum of all payloads):        60
After within-payload dedup:                   60 (-0)
After cross-payload dedup:                    60 (-0)

Total removed: 0 (0.0%)

Breakdown by commodity:
  corn     ->     30 →     30 (removed    0,   0.0%)
  wheat    ->     30 →     30 (removed    0,   0.0%)


In [21]:
# Export consolidated clean payload
output_path = DATA_DIR / 'consolidated_commodities_clean.csv'

df_consolidated.to_csv(output_path, index=False)
print(f"\n✓ Exported clean payload to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"  Records: {len(df_consolidated)}")
print(f"  Columns: {len(df_consolidated.columns)}")


✓ Exported clean payload to: data/consolidated_commodities_clean.csv
  File size: 27.1 KB
  Records: 60
  Columns: 15


In [22]:
# Preview clean payload
print(f"\nClean Payload Preview:")
print(f"\nColumns: {list(df_consolidated.columns)}")
print(f"\nFirst 5 rows:")
print(df_consolidated.head().to_string())

print(f"\nRecord counts by commodity:")
print(df_consolidated['commodity'].value_counts().sort_index())


Clean Payload Preview:

Columns: ['name', 'content', 'smlID', 'authorID', 'authorName', 'authorType', 'date', 'dateTimestamp', 'dataID', 'link', 'isEditorPick', 'image', 'searchable', 'prioirty', 'commodity']

First 5 rows:
                             name                                                                                                                                                                                                      content smlID   authorID      authorName authorType       date  dateTimestamp  dataID                                                          link  isEditorPick                                                                                         image      searchable  prioirty commodity
0          THE MARKET WORD - CORN  Seasonally, this is the time frame to follow the grain markets! The July Corn contract is going into a weather driven market condition. Any dry weather or flooding could severely damage the Corn crop....   417  101024